# G2.1 — Storage tank energy balance (guided)

Exercise card: `G2.1_storage_energy_balance.md` · 15 min · prerequisites: lesson §5
(open-system energy balance, M03 node equations).

**Goal.** Write the energy balance of each node of an open system with a through-flow, and verify
that the tank as a whole conserves energy.

Parameters (illustrative): tank $V$ = 200 L, $N$ = 5 nodes, $UA_{loss}$ = 1.5 W/K, no conduction
between nodes, water $\rho$ = 995.586 kg/m³, $c_p$ = 4184 J/(kg·K). The tank starts uniformly at
25 °C; from $t = 0$ water at 55 °C enters at the top with $\dot m$ = 0.05 kg/s and leaves at the
bottom; surroundings at 20 °C. Simulate 3 h.

**How to work.** Run the cells from the top (*Shift+Enter*). Complete the cells marked **TODO** and write your answers in the cells marked *Your answer*. Before the self-check, run **Kernel → Restart Kernel and Run All Cells…**: the notebook must run from top to bottom.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.integrate import solve_ivp

from energy_training.plotting.style import apply_style, degc, hours
from energy_training.simulation import storage as st
from energy_training.simulation.storage import CP_WATER, RHO_WATER
from energy_training.utilities import celsius_to_kelvin

apply_style()
H = 3600.0  # s per hour

V = 0.2  # m3 (illustrative)
N = 5  # nodes, node 1 at the top
UA_LOSS = 1.5  # W/K (illustrative)
M_FLOW = 0.05  # kg/s, enters at the top
T_IN = float(celsius_to_kelvin(55.0))  # K
T_ENV = float(celsius_to_kelvin(20.0))  # K
T_START = float(celsius_to_kelvin(25.0))  # K
DURATION = 3 * H

## 1. By hand

Mass and heat capacity of one node, and the mean residence time $\rho V/\dot m$.

*Your answer:*

## 2. Node equations

Complete `rhs(t, T, n)`: the energy balance of each of the `n` nodes (node 1 at the top),

$$m_{node}\,c_p\,\frac{dT_k}{dt} = \dot m\,c_p\,(T_{k-1} - T_k) - \frac{UA_{loss}}{n}\,(T_k - T_{env}),
\qquad T_0 = T_{in}.$$

In [ ]:
def rhs(t, T, n=N):
    """Return dT/dt [K/s] for the n node temperatures T (top first)."""
    # TODO: m_node cp dT_k/dt = m_flow cp (T_up - T_k) - UA_loss/n (T_k - T_env)
    raise NotImplementedError("TODO: complete rhs()")

## 3. Simulate

`scipy.integrate.solve_ivp` with `rtol = atol = 1e-8`, output every 60 s (provided).

In [ ]:
def simulate(n=N, f=None):
    """Node temperatures T[k, j] at the times t[k] for n nodes; f(t, T, n) defaults to rhs."""
    t_eval = np.arange(0.0, DURATION + 1.0, 60.0)
    sol = solve_ivp(
        f or rhs,
        (0.0, DURATION),
        np.full(n, T_START),
        t_eval=t_eval,
        rtol=1e-8,
        atol=1e-8,
        args=(n,),
    )
    return sol.t, sol.y.T


t, T = simulate()
fig, ax = plt.subplots()
for j in range(N):
    ax.plot(hours(t), degc(T[:, j]), label=f"node {j + 1}")
ax.set(xlabel="time [h]", ylabel="T [°C]")
ax.legend();

## 4. Energy balance

Complete `energy_terms`: over the 3 h, the energy **stored** in the tank, the energy **brought in
by the flow** (in at $T_{in}$, out at the temperature of the last node) and the energy **lost**
through the insulation. Use `np.trapezoid` for the integrals.

In [ ]:
def energy_terms(t, T, n=N):
    """Return (stored, advected, lost) in J for the sampled solution T of shape (len(t), n)."""
    # TODO: use np.trapezoid for the integrals
    raise NotImplementedError("TODO: complete energy_terms()")

In [ ]:
stored, advected, lost = energy_terms(t, T)
residual = stored - (advected - lost)
residual_rel = abs(residual) / (abs(stored) + abs(advected) + abs(lost))
print(
    f"stored {stored / 3.6e6:.4f} kWh, advected {advected / 3.6e6:.4f} kWh, "
    f"lost {lost / 3.6e6:.4f} kWh"
)
print(f"residual {residual:.1f} J = {residual_rel:.1e} of the throughput")

## Self-check (and step 5)

Compares your `rhs` with `energy_training.simulation.storage.rhs` at a few random states, and
checks the expected outcome of the card: residual below $10^{-4}$ of the throughput.

In [ ]:
p = st.StorageParams(V=V, N=N, UA_loss=UA_LOSS)
rng = np.random.default_rng(1)
for x in 290 + 40 * rng.random((5, N)):
    diff = np.max(np.abs(rhs(0.0, x) - st.rhs(0.0, x, M_FLOW, T_IN, T_ENV, p)))
    assert diff < 1e-12, f"rhs differs from the package by {diff:.2e} K/s at T = {x}"
assert residual_rel < 1e-4, "energy-balance residual above 1e-4: check energy_terms"
print("Self-check passed.")

## 6. What does $N$ change?

The lesson claims that $N$ changes the *shape* of the outlet front but not the mean residence time.
**Before running**: does the outlet reach 50 °C sooner or later with more nodes? Write your
prediction. Then simulate $N$ = 1, 5 and 20, find when the outlet reaches 50 °C, and plot the
three outlet temperatures.

*Your prediction:*

In [ ]:
# TODO: for n in (1, 5, 20): simulate(n), time at which the outlet (last node) reaches 50 degC,
#       plot the outlet temperatures

*Your answer:*

## Extension: conduction between nodes

Add conduction between neighbouring nodes ($G$ = 2 W/K, use `st.rhs` with
`st.StorageParams(..., G=2.0)`). Show that the energy balance of the tank does not change, and
explain why the conduction terms cancel in the sum over the nodes.

In [ ]:
# TODO (optional)

*Your answer:*